### 3. Data Transformation

#### Load the cleaned datasets

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data/processed"

nasa = pd.read_csv(
    PROCESSED_DIR / "nasa_power_clean.csv"
)

open_meteo = pd.read_csv(
    PROCESSED_DIR / "open_meteo_clean.csv"
)

locations = pd.read_csv(
    PROCESSED_DIR / "locations_clean.csv"
)

nasa["date"] = pd.to_datetime(nasa["date"])
open_meteo["date"] = pd.to_datetime(open_meteo["date"])

print("NASA:", nasa.shape)
print("Open-Meteo:", open_meteo.shape)
print("Locations:", locations.shape)

NASA: (5117, 10)
Open-Meteo: (5117, 11)
Locations: (7, 8)


#### Rename NASA POWER columns

In [2]:
nasa.columns.tolist()

['date',
 'T2M',
 'RH2M',
 'WS2M',
 'ALLSKY_SFC_SW_DWN',
 'location',
 'latitude',
 'longitude',
 'state',
 'region']

In [3]:
nasa = nasa.rename(columns={
    "T2M": "temperature",
    "RH2M": "relative_humidity",
    "WS2M": "wind_speed",
    "ALLSKY_SFC_SW_DWN": "solar_radiation"
})

In [4]:
nasa.head()

,date,temperature,relative_humidity,wind_speed,solar_radiation,location,latitude,longitude,state,region
0,2023-01-01,13.38,48.20,0.96,3.0706,Delhi,28.6139,77.209,Delhi,North
1,2023-01-02,11.70,52.39,1.16,1.8490,Delhi,28.6139,77.209,Delhi,North
2,2023-01-03,9.28,56.00,1.76,1.9087,Delhi,28.6139,77.209,Delhi,North
3,2023-01-04,8.71,59.91,1.50,1.9310,Delhi,28.6139,77.209,Delhi,North
4,2023-01-05,8.75,58.49,1.52,1.7275,Delhi,28.6139,77.209,Delhi,North


#### Rename Open-Meteo columns

In [5]:
open_meteo = open_meteo.rename(columns={
    "temperature_2m_max": "max_temperature",
    "temperature_2m_min": "min_temperature",
    "temperature_2m_mean": "mean_temperature",
    "precipitation_sum": "precipitation",
    "wind_speed_10m_max": "max_wind_speed"
})

In [6]:
open_meteo.head()

,date,max_temperature,min_temperature,mean_temperature,precipitation,max_wind_speed,location,latitude,longitude,state,region
0,2023-01-01,18.3,7.9,12.5,0.0,8.7,Delhi,28.6139,77.209,Delhi,North
1,2023-01-02,18.2,6.3,12.2,0.0,9.2,Delhi,28.6139,77.209,Delhi,North
2,2023-01-03,16.6,6.0,10.6,0.0,9.7,Delhi,28.6139,77.209,Delhi,North
3,2023-01-04,16.5,5.4,9.7,0.0,11.2,Delhi,28.6139,77.209,Delhi,North
4,2023-01-05,15.4,5.4,8.9,0.0,10.2,Delhi,28.6139,77.209,Delhi,North


#### Add date attributes

In [7]:
for df in [nasa, open_meteo]:
    df["year"] = df["date"].dt.year
    df["quarter"] = df["date"].dt.quarter
    df["month"] = df["date"].dt.month
    df["month_name"] = df["date"].dt.month_name()
    df["day"] = df["date"].dt.day

In [8]:
open_meteo[
    ["date", "year", "quarter", "month", "month_name", "day"]
].head()

,date,year,quarter,month,month_name,day
0,2023-01-01,2023,1,1,January,1
1,2023-01-02,2023,1,1,January,2
2,2023-01-03,2023,1,1,January,3
3,2023-01-04,2023,1,1,January,4
4,2023-01-05,2023,1,1,January,5


#### Merge NASA + Open-Meteo

In [9]:
weather = pd.merge(
    open_meteo,
    nasa[
        [
            "date",
            "location",
            "temperature",
            "relative_humidity",
            "wind_speed",
            "solar_radiation"
        ]
    ],
    on=["date", "location"],
    how="inner"
)

In [10]:
print(weather.shape)
weather.head()

(5117, 20)


,date,max_temperature,min_temperature,mean_temperature,precipitation,max_wind_speed,location,latitude,longitude,state,region,year,quarter,month,month_name,day,temperature,relative_humidity,wind_speed,solar_radiation
0,2023-01-01,18.3,7.9,12.5,0.0,8.7,Delhi,28.6139,77.209,Delhi,North,2023,1,1,January,1,13.38,48.20,0.96,3.0706
1,2023-01-02,18.2,6.3,12.2,0.0,9.2,Delhi,28.6139,77.209,Delhi,North,2023,1,1,January,2,11.70,52.39,1.16,1.8490
2,2023-01-03,16.6,6.0,10.6,0.0,9.7,Delhi,28.6139,77.209,Delhi,North,2023,1,1,January,3,9.28,56.00,1.76,1.9087
3,2023-01-04,16.5,5.4,9.7,0.0,11.2,Delhi,28.6139,77.209,Delhi,North,2023,1,1,January,4,8.71,59.91,1.50,1.9310
4,2023-01-05,15.4,5.4,8.9,0.0,10.2,Delhi,28.6139,77.209,Delhi,North,2023,1,1,January,5,8.75,58.49,1.52,1.7275


In [11]:
weather[["date", "location"]].duplicated().sum()

np.int64(0)

#### Create a unified average temperature

In [12]:
weather["average_temperature"] = weather[
    ["mean_temperature", "temperature"]
].mean(axis=1)

#### Create temperature range

In [13]:
weather["temperature_range"] = (
    weather["max_temperature"]
    - weather["min_temperature"]
)

#### Create a temperature anomaly

In [14]:
location_baseline = (
    weather
    .groupby("location")["max_temperature"]
    .mean()
    .rename("baseline_max_temperature")
    .reset_index()
)

In [15]:
weather = weather.merge(
    location_baseline,
    on="location",
    how="left"
)

In [16]:
weather["temperature_anomaly"] = (
    weather["max_temperature"]
    - weather["baseline_max_temperature"]
)

#### Heatwave Classification

In [28]:
# Initialize heatwave classification
weather["heatwave_flag"] = 0
weather["severity"] = "Normal"
weather["severity_level"] = 0

In [29]:
# Create location-month baseline for maximum temperature

monthly_baseline = (
    weather
    .groupby(["location", "month"])["max_temperature"]
    .mean()
    .rename("monthly_baseline_max_temperature")
    .reset_index()
)

weather = weather.merge(
    monthly_baseline,
    on=["location", "month"],
    how="left"
)

# Calculate temperature anomaly

weather["temperature_anomaly"] = (
    weather["max_temperature"]
    - weather["monthly_baseline_max_temperature"]
)

In [30]:
# Heatwave classification

heatwave_condition = (
    (weather["max_temperature"] >= 40) &
    (weather["temperature_anomaly"] >= 4.5)
)

severe_heatwave_condition = (
    (weather["max_temperature"] >= 40) &
    (weather["temperature_anomaly"] >= 6.5)
)

actual_heatwave = weather["max_temperature"] >= 45
actual_severe_heatwave = weather["max_temperature"] >= 47

# Mumbai coastal condition
mumbai_heatwave = (
    (weather["location"] == "Mumbai") &
    (weather["max_temperature"] >= 37) &
    (weather["temperature_anomaly"] >= 4.5)
)

mumbai_severe_heatwave = (
    (weather["location"] == "Mumbai") &
    (weather["max_temperature"] >= 37) &
    (weather["temperature_anomaly"] >= 6.5)
)

# Severe heatwave first
severe_condition = (
    severe_heatwave_condition |
    actual_severe_heatwave |
    mumbai_severe_heatwave
)

weather.loc[severe_condition, "heatwave_flag"] = 1
weather.loc[severe_condition, "severity"] = "Severe Heatwave"
weather.loc[severe_condition, "severity_level"] = 2

# Normal heatwave
heatwave_condition_final = (
    heatwave_condition |
    actual_heatwave |
    mumbai_heatwave
)

weather.loc[
    heatwave_condition_final & ~severe_condition,
    "heatwave_flag"
] = 1

weather.loc[
    heatwave_condition_final & ~severe_condition,
    "severity"
] = "Heatwave"

weather.loc[
    heatwave_condition_final & ~severe_condition,
    "severity_level"
] = 1

In [31]:
weather["severity"].value_counts()

severity
Normal             5053
Heatwave             56
Severe Heatwave       8
Name: count, dtype: int64

In [32]:
weather.groupby(["location", "severity"]).size()

location   severity       
Bengaluru  Normal             731
Delhi      Heatwave            18
           Normal             707
           Severe Heatwave      6
Guwahati   Normal             731
Jaipur     Heatwave            17
           Normal             713
           Severe Heatwave      1
Kolkata    Heatwave             6
           Normal             725
Mumbai     Heatwave             6
           Normal             724
           Severe Heatwave      1
Nagpur     Heatwave             9
           Normal             722
dtype: int64

In [33]:
weather[
    weather["heatwave_flag"] == 1
][[
    "date",
    "location",
    "max_temperature",
    "temperature_anomaly",
    "severity"
]].head(20)

,date,location,max_temperature,temperature_anomaly,severity
142,2023-05-23,Delhi,43.6,4.603226,Heatwave
502,2024-05-17,Delhi,44.0,5.003226,Heatwave
503,2024-05-18,Delhi,44.0,5.003226,Heatwave
504,2024-05-19,Delhi,44.9,5.903226,Heatwave
505,2024-05-20,Delhi,43.6,4.603226,Heatwave
510,2024-05-25,Delhi,44.4,5.403226,Heatwave
511,2024-05-26,Delhi,46.0,7.003226,Severe Heatwave
512,2024-05-27,Delhi,45.7,6.703226,Severe Heatwave
513,2024-05-28,Delhi,45.8,6.803226,Severe Heatwave
514,2024-05-29,Delhi,45.7,6.703226,Severe Heatwave


#### Add region/state information

In [18]:
location_info = locations[
    [
        "location",
        "city",
        "state",
        "osm_latitude",
        "osm_longitude"
    ]
].copy()

In [19]:
location_info = location_info.rename(columns={
    "osm_latitude": "latitude",
    "osm_longitude": "longitude"
})

In [20]:
weather = weather.merge(
    location_info,
    on="location",
    how="left"
)

In [21]:
weather.head()

,date,max_temperature,min_temperature,mean_temperature,precipitation,max_wind_speed,location,latitude_x,longitude_x,state_x,...,average_temperature,temperature_range,baseline_max_temperature,temperature_anomaly,heatwave_flag,severity,city,state_y,latitude_y,longitude_y
0,2023-01-01,18.3,7.9,12.5,0.0,8.7,Delhi,28.6139,77.209,Delhi,...,12.940,10.4,30.397674,-12.097674,0,Normal,Delhi,Delhi,28.632803,77.219771
1,2023-01-02,18.2,6.3,12.2,0.0,9.2,Delhi,28.6139,77.209,Delhi,...,11.950,11.9,30.397674,-12.197674,0,Normal,Delhi,Delhi,28.632803,77.219771
2,2023-01-03,16.6,6.0,10.6,0.0,9.7,Delhi,28.6139,77.209,Delhi,...,9.940,10.6,30.397674,-13.797674,0,Normal,Delhi,Delhi,28.632803,77.219771
3,2023-01-04,16.5,5.4,9.7,0.0,11.2,Delhi,28.6139,77.209,Delhi,...,9.205,11.1,30.397674,-13.897674,0,Normal,Delhi,Delhi,28.632803,77.219771
4,2023-01-05,15.4,5.4,8.9,0.0,10.2,Delhi,28.6139,77.209,Delhi,...,8.825,10.0,30.397674,-14.997674,0,Normal,Delhi,Delhi,28.632803,77.219771


In [22]:
print("Rows:", len(weather))
print("Columns:", len(weather.columns))

Rows: 5117
Columns: 30


In [23]:
weather.isnull().sum()

date                        0
max_temperature             0
min_temperature             0
mean_temperature            0
precipitation               0
max_wind_speed              0
location                    0
latitude_x                  0
longitude_x                 0
state_x                     0
region                      0
year                        0
quarter                     0
month                       0
month_name                  0
day                         0
temperature                 0
relative_humidity           0
wind_speed                  0
solar_radiation             0
average_temperature         0
temperature_range           0
baseline_max_temperature    0
temperature_anomaly         0
heatwave_flag               0
severity                    0
city                        0
state_y                     0
latitude_y                  0
longitude_y                 0
dtype: int64

In [24]:
weather.duplicated(
    subset=["date", "location"]
).sum()

np.int64(0)

In [25]:
weather.groupby("location").size()

location
Bengaluru    731
Delhi        731
Guwahati     731
Jaipur       731
Kolkata      731
Mumbai       731
Nagpur       731
dtype: int64

#### Save the transformed master dataset

In [34]:
TRANSFORMED_FILE = (
    PROCESSED_DIR / "weather_transformed.csv"
)

weather.to_csv(
    TRANSFORMED_FILE,
    index=False
)

print(f"Saved to: {TRANSFORMED_FILE}")

Saved to: D:\College\SEM 5\MDA\heatwave_datawarehouse\data\processed\weather_transformed.csv


In [35]:
print("Total records:", len(weather))
print("Heatwave days:", weather["heatwave_flag"].sum())
print("Missing values:", weather.isnull().sum().sum())

Total records: 5117
Heatwave days: 64
Missing values: 0
